# Tests for the UTR-2 transient service

MMODA runs notebooks named `test_*.ipynb` to check that a workflow still
behaves, and excludes them from becoming services themselves. Each cell below
raises `AssertionError` on failure, which is what marks the test as failed.

These run the service through `nb2workflow.nbadapter.run` — the same path MMODA
uses — so they exercise **output gathering**, not just execution. That matters:
a notebook can run perfectly and still deliver no products, because outputs are
collected afterwards by scrapbook. That is exactly the bug this file was written
to catch (see README.md, 'Open problems and caveats', item 12).

The equivalent checks also exist as ordinary pytest tests in
`tests/test_notebook.py`, which is the more convenient form during development.


## Setup

In [ ]:
import os

import nb2workflow.nbadapter as nbadapter
from nb2workflow.nbadapter import run

# nb2workflow's health probe calls os.statvfs, which does not exist on Windows.
# It is only used for logging, so stub it out and let these tests run anywhere.
if not hasattr(os, "statvfs"):
    nbadapter.current_health = lambda: {}

# Resolve the service notebook from the installed package rather than from the
# working directory, which the runner does not guarantee.
import src

NOTEBOOK = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(src.__file__))),
                        "mmoda", "utr2_transients.ipynb")
assert os.path.exists(NOTEBOOK), f"service notebook not found at {NOTEBOOK}"

EXPECTED_OUTPUTS = {"transient_table", "sky_map", "histograms", "query_summary"}
print("testing", NOTEBOOK)


## Whole-sky query

In [ ]:
# The catalogue is already pre-filtered, so a default whole-sky query returns
# every one of the 380 transients.
result = run(NOTEBOOK, {"radius": 180.0, "snr_threshold": 8.0})

missing = EXPECTED_OUTPUTS - set(result)
assert not missing, f"MMODA would receive no {missing}"

assert "380 of 380" in result["query_summary"], result["query_summary"]
print("whole sky OK:", result["query_summary"][:80])


## Cone search

In [ ]:
# A 20 degree cone around Cas A selects 10 transients.
result = run(NOTEBOOK, {"src_name": "Cas A", "RA": 350.85, "DEC": 58.815,
                        "radius": 20.0, "snr_threshold": 8.0})

assert "10 of 380" in result["query_summary"], result["query_summary"]
assert "within 20 deg" in result["query_summary"], result["query_summary"]
print("cone search OK:", result["query_summary"][:80])


## Products are actually gathered

In [ ]:
# The table must survive serialisation, and both figures must arrive as real
# base64 payloads rather than bare filenames.
result = run(NOTEBOOK, {"src_name": "Cas A", "RA": 350.85, "DEC": 58.815,
                        "radius": 20.0, "snr_threshold": 8.0})

assert result["transient_table"], "the astropy table came back empty"

for name in ("sky_map", "histograms"):
    content = result.get(f"{name}_content")
    assert content, f"{name} produced no file content"
    assert len(content) > 1000, f"{name} content is implausibly small"

print("products OK:",
      {k: len(str(v)) for k, v in result.items() if k.endswith("_content")})


## Empty results explain themselves

In [ ]:
# An impossible query must fail with our explanation, not a bare traceback.
#
# Note: run() cannot be used here. It ignores what execute() returns, so a
# failed workflow comes back from run() as an empty dict with no exception
# raised at all. MMODA itself inspects the exceptions from execute(), which is
# how the message reaches the user, so that is what we assert against.
nba = nbadapter.NotebookAdapter(NOTEBOOK)
exceptions = nba.execute({"radius": 180.0, "snr_threshold": 10000.0},
                         log_output=False, progress_bar=False)

assert exceptions, "an impossible query unexpectedly succeeded"
assert "No transients" in str(exceptions), f"unhelpful failure message: {exceptions}"
print("failure path OK - the user is told why nothing matched")


In [ ]:
print("All UTR-2 transient service tests passed.")
